# Mini MapReduce in Python
Built from scratch: no Hadoop, no Spark, no `Counter`, no `groupby`.

**Pipeline:** Input → Map (threads) → Combine → Partition (2 reducers) → Shuffle & Sort → Reduce → Output

## Cell 1: Imports and settings

In [1]:
import os
import re
import threading

NUM_REDUCERS = 2          # the brief asks for 2 reducers
INPUT_FOLDER = "claims"   # one .txt file = one split
OUTPUT_FOLDER = "output"  # one output file per reducer

## Cell 2: Create sample input (the 3 ETI claim notes)
Skip this cell if you already have your own `claims/` folder.

In [2]:
os.makedirs(INPUT_FOLDER, exist_ok=True)

samples = {
    "claim1.txt": "car damage theft",
    "claim2.txt": "car damage water",
    "claim3.txt": "car damage fire roof",
}
for name, text in samples.items():
    with open(os.path.join(INPUT_FOLDER, name), "w", encoding="utf-8") as f:
        f.write(text)

print("Created:", sorted(os.listdir(INPUT_FOLDER)))

Created: ['claim1.txt', 'claim2.txt', 'claim3.txt']


## Cell 3: INPUT phase
Each text file is one **split**.

In [3]:
def read_splits(folder):
    splits = []
    for name in sorted(os.listdir(folder)):
        path = os.path.join(folder, name)
        if os.path.isfile(path) and name.endswith(".txt"):
            with open(path, encoding="utf-8") as f:
                splits.append((name, f.read()))
    return splits

splits = read_splits(INPUT_FOLDER)
for name, text in splits:
    print(f"{name}: {text!r}")

claim1.txt: 'car damage theft'
claim2.txt: 'car damage water'
claim3.txt: 'car damage fire roof'


## Cell 4: MAP phase
For every word emit `(word, 1)`.

In [4]:
def map_fn(text):
    pairs = []
    for word in re.findall(r"[a-z']+", text.lower()):
        pairs.append((word, 1))
    return pairs

# quick test
print(map_fn("Car damage, car THEFT"))

[('car', 1), ('damage', 1), ('car', 1), ('theft', 1)]


## Cell 5: COMBINE phase
Local subtotals on the mapper, using a hand-made dictionary.

In [5]:
def combine(pairs):
    local = {}
    for word, n in pairs:
        if word in local:
            local[word] += n
        else:
            local[word] = n
    return local

# quick test
print(combine(map_fn("car car damage")))

{'car': 2, 'damage': 1}


## Cell 6: PARTITION phase
Decides which reducer gets each word. Deterministic: the same word always goes to the same reducer.

In [6]:
def partition(word):
    return sum(ord(c) for c in word) % NUM_REDUCERS

for w in ["car", "damage", "fire", "roof", "theft", "water"]:
    print(w, "-> reducer", partition(w))

car -> reducer 0
damage -> reducer 1
fire -> reducer 0
roof -> reducer 0
theft -> reducer 1
water -> reducer 1


## Cell 7: Mapper task (runs inside a thread)
Map → Combine → Partition for one split. The lock protects the shared results list.

In [7]:
def mapper_task(name, text, results, lock):
    pairs = map_fn(text)                         # map
    local = combine(pairs)                       # combine
    buckets = [[] for _ in range(NUM_REDUCERS)]  # one bucket per reducer
    for word, n in local.items():                # partition
        buckets[partition(word)].append((word, n))
    with lock:
        results.append(buckets)
    print(f"[mapper:{name}] {len(pairs)} pairs -> {len(local)} after combine")

## Cell 8: SHUFFLE & SORT phase
Collect one reducer's data from **all** mappers, group by word, sort by key.

In [8]:
def shuffle_and_sort(all_buckets, r):
    groups = {}
    for buckets in all_buckets:
        for word, n in buckets[r]:
            if word not in groups:
                groups[word] = []
            groups[word].append(n)
    return sorted(groups.items())

## Cell 9: REDUCE phase
Sum the list of counts for one word.

In [9]:
def reduce_fn(word, counts):
    total = 0
    for c in counts:
        total += c
    return word, total

print(reduce_fn("car", [1, 1, 1]))

('car', 3)


## Cell 10: Run the whole job
All mappers start at the same time, `join()` waits for all of them, then the reducers run.

In [10]:
def run_job():
    splits = read_splits(INPUT_FOLDER)
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    results, lock, threads = [], threading.Lock(), []

    # run mappers concurrently, one thread per split
    for name, text in splits:
        t = threading.Thread(target=mapper_task, args=(name, text, results, lock))
        threads.append(t)
        t.start()
    for t in threads:
        t.join()

    # shuffle & sort + reduce, one output file per reducer
    final = {}
    for r in range(NUM_REDUCERS):
        grouped = shuffle_and_sort(results, r)
        path = os.path.join(OUTPUT_FOLDER, f"part-r-{r:05d}.txt")
        with open(path, "w", encoding="utf-8") as f:
            for word, counts in grouped:
                w, total = reduce_fn(word, counts)
                f.write(f"{w}\t{total}\n")
                final[w] = total
        print(f"[reducer:{r}] {len(grouped)} words -> {path}")
    return final

final = run_job()

[mapper:claim1.txt] 3 pairs -> 3 after combine
[mapper:claim2.txt] 3 pairs -> 3 after combine
[mapper:claim3.txt] 4 pairs -> 4 after combine
[reducer:0] 3 words -> output\part-r-00000.txt
[reducer:1] 3 words -> output\part-r-00001.txt


## Cell 11: Show the output files

In [11]:
for fname in sorted(os.listdir(OUTPUT_FOLDER)):
    print(f"--- {fname} ---")
    with open(os.path.join(OUTPUT_FOLDER, fname), encoding="utf-8") as f:
        print(f.read())

--- part-r-00000.txt ---
car	3
fire	1
roof	1

--- part-r-00001.txt ---
damage	3
theft	1
water	1



## Cell 12: Self-check
Expected from the lab: car 3, damage 3, fire 1, roof 1, theft 1, water 1.

In [12]:
expected = {"car": 3, "damage": 3, "fire": 1, "roof": 1, "theft": 1, "water": 1}
print("Result  :", dict(sorted(final.items())))
print("Expected:", expected)
print("PASS" if final == expected else "FAIL")

Result  : {'car': 3, 'damage': 3, 'fire': 1, 'roof': 1, 'theft': 1, 'water': 1}
Expected: {'car': 3, 'damage': 3, 'fire': 1, 'roof': 1, 'theft': 1, 'water': 1}
PASS
